In [1]:
%pip install uv --quiet
!uv pip install pandas numpy plotly matplotlib
!uv sync

Note: you may need to restart the kernel to use updated packages.


Using Python 3.12.3 environment at: C:\Users\logan\OneDrive\Documents\SeniorSpring\Adv Data Sci\Modern-Store-Of-Value\.venv
Checked 4 packages in 341ms
Resolved 147 packages in 32ms
Checked 143 packages in 395ms


## Combine csvs

In [10]:
import pandas as pd
import os

BASE_DIR = os.path.abspath("../data")

FILES = {
    "sortino":             os.path.join(BASE_DIR, "sortino_metric.csv"),
    "calmar":              os.path.join(BASE_DIR, "calmar_metrics.csv"),
    "inflation":           os.path.join(BASE_DIR, "inflation_metrics.csv"),
    "market_independence": os.path.join(BASE_DIR, "market_independence_metrics.csv"),
    "crisis":              os.path.join(BASE_DIR, "crisis_metrics.csv"),
}

def load(name: str, path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    if "Asset_Name" in df.columns:
        df = df.rename(columns={"Asset_Name": "Ticker"})

    rename_map = {
        col: f"{name}__{col}"
        for col in df.columns
        if col != "Ticker"
    }
    return df.rename(columns=rename_map)


dfs = [load(name, path) for name, path in FILES.items()]

merged = dfs[0]
for df in dfs[1:]:
    merged = merged.merge(df, on="Ticker", how="outer")

merged = merged.sort_values("Ticker").reset_index(drop=True)

# ── Consolidate Category columns into one ─────────────────────────────────────
category_cols = [col for col in merged.columns if col.endswith("__Category")]
merged["Category"] = merged[category_cols].bfill(axis=1).iloc[:, 0]
merged = merged.drop(columns=category_cols)

# Move Category to the front, right after Ticker
cols = ["Ticker", "Category"] + [c for c in merged.columns if c not in ("Ticker", "Category")]
merged = merged[cols]

# ── Save ──────────────────────────────────────────────────────────────────────
output_path = os.path.join(BASE_DIR, "merged_metrics.csv")
merged.to_csv(output_path, index=False)

print(f"✅  Merged {len(merged)} tickers × {len(merged.columns)} columns")
print(f"    Saved → {output_path}\n")
print(merged.to_string(max_rows=10))

✅  Merged 23 tickers × 16 columns
    Saved → c:\Users\logan\OneDrive\Documents\SeniorSpring\Adv Data Sci\Modern-Store-Of-Value\data\merged_metrics.csv

   Ticker                      Category  sortino__Sortino_Ratio  sortino__Normalized_Sortino_Score_1_10  calmar__Raw_Calmar_Ratio  calmar__Normalized_Calmar_Score_1_10  inflation__Raw_Real_Return_%  inflation__Normalized_Inflation_Score_1_10  market_independence__Raw_Correlation  market_independence__Raw_Beta  market_independence__Correlation_MinMax  market_independence__Beta_MinMax  market_independence__Correlation_Percentile  market_independence__Beta_Percentile  crisis__Raw_Mean_Stress_Score  crisis__Normalized_Crisis_Score_1_10
0    AAPL             Individual Stocks                  1.0685                                    6.73                  0.532745                                  7.55                     72.214169                                        7.95                                0.6545                         1.034